# ESS (Environmental Sensors Support) Data Usage Example - MTMount accelerometers

Querying MTMount accelerometers

This notebook shows how to retrieve, process, and visualize acceleration data from the MTMount accelerometers. The data is extracted from multiple sensors.

In [ ]:
import asyncio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
import time
import warnings

from astropy.time import Time, TimeDelta
from lsst.summit.utils.efdUtils import makeEfdClient
from lsst_efd_client.efd_helper import merge_packed_time_series

# Ignore the many warning messages from ``merge_packed_time_series``
#warnings.simplefilter(action="ignore", category=FutureWarning)

## Initialize EFD Client

In [ ]:
# Create an instance of the EFD client
client = makeEfdClient()

## Define Time Range

The time is defined using the `Time` class from `astropy`, which follows the ISO 8601 format:  

**YYYY-MM-DD HH:MM:SSZ**, where:  
- `YYYY` is the year  
- `MM` is the month  
- `DD` is the day  
- `HH:MM:SS` represents hours, minutes, and seconds in 24-hour format  
- `Z` indicates that the time is in Coordinated Universal Time (UTC)  

In this case, the selected range is from 08:00 to 08:10 UTC on March 22, 2023.

In [ ]:
# Define the time range using UTC format
start = Time("2023-03-22 08:00:00Z", scale="utc")
end = Time("2023-03-22 08:10:00Z", scale="utc")

## Define Base Fields and Sensor Names  

In [ ]:
# Define the fields related to acceleration along different axes
base_fields = ["accelerationX", "accelerationY", "accelerationZ"]

# List of sensor names
sensor_names = [
    "SST top end ring +x -y",  
    "SST top end ring -x -y",  
    "SST spider spindle",  
    "SST M2 surrogate"  
]

##  Get all of the data for the selected times

In [ ]:
# Fetch accelerometer data from the EFD for the given time range
packed_dataframe = await client.select_time_series(
    "lsst.sal.ESS.accelerometer",  # Topic containing accelerometer telemetry data
    ["*"],  # Select all available fields
    start,  # Start time for data retrieval
    end    # End time for data retrieval
)

## Plot Accelerometer Data for Each Sensor  

In [ ]:
# Create a 2x2 grid for plotting acceleration data
fig, axs = plt.subplots(2, 2, figsize=(8, 8))
plt.subplots_adjust(hspace=0.5, wspace=0.5)

# Loop through each sensor and plot its acceleration data
for i, sensor_name in enumerate(sensor_names):
    # Filter data for the current sensor
    sub_dataframe = packed_dataframe.loc[packed_dataframe.sensorName == sensor_name]

    # Determine subplot position
    plot_x = i % 2
    plot_y = int(i / 2)
    ax = axs[plot_x][plot_y]
    
    # Set subplot title and labels
    ax.set_title(sensor_name)
    ax.set_ylabel("Acceleration (m/s²)")
    
    # Loop through acceleration components (X, Y, Z)
    for base_field in base_fields:
        df = merge_packed_time_series(
            sub_dataframe,
            base_field,
            stride=1,
            ref_timestamp_col="timestamp"
        )
        df[base_field].plot(ax=ax, label=base_field[-1])  # Use last character of field name as label
    
    ax.legend() 
